#injesting the data in

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, DecimalType
from pyspark.sql.functions import lit
from datetime import datetime
import yaml

#reading the data in df

In [0]:

with open("../config/pipeline_config.yaml") as pc:
  config = yaml.safe_load(pc)

accounts_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_ref", StringType(), False),
    StructField("account_type", StringType(), False),
    StructField("account_status", StringType(), False),
    StructField("open_date", StringType(), False),
    StructField("product_tier", StringType(), False),
    StructField("mobile_number", StringType(), True),
    StructField("digital_channel", StringType(), False),
    StructField("credit_limit", DoubleType(), True),
    StructField("current_balance", DoubleType(), False),
    StructField("last_activity_date", StringType(), True)])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("id_number", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("dob", StringType(), False),
    StructField("gender", StringType(), False),
    StructField("province", StringType(), False),
    StructField("income_band", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("risk_score", IntegerType(), False),
    StructField("kyc_status", StringType(), False),
    StructField("product_flags", StringType(), False)
  ])

trasactions_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("transaction_date", StringType(), False),
    StructField("transaction_time", StringType(), False),
    StructField("transaction_type", StringType(), False),
    StructField("merchant_category", StringType(), True),
    StructField("merchant_subcategory", StringType(), True),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("location", StringType([
        StructField("province", StringType(), True),
        StructField("city", StringType(), True),
        StructField("coordinates", StringType(), True)
    ]), True),
    StructField("metadata", StringType([
        StructField("device_id", StringType(), True),
        StructField("session_id", StringType(), True),
        StructField("retry_flag", BooleanType(), True),
    ]), True)
  ])  

schemas = {
    "accounts": {
      "schema" : accounts_schema,
      "extension" : config["input"]["accounts_path"].split(".")[-1]
    },
    "customers": {
      "schema" : customers_schema,
      "extension" : config["input"]["customers_path"].split(".")[-1]
    },
    "transactions": {
      "schema" : trasactions_schema,
      "extension" : config["input"]["transactions_path"].split(".")[-1]
    }
}

run_timestamp = datetime.now()

for source, props in schemas.items():
  if props["extension"] == "csv":
    df = spark.read.csv(config["input"][f"{source}_path"], header=True, schema=props["schema"])
  elif props["extension"] == "jsonl" :
    df = spark.read.json(config["input"][f"{source}_path"], schema=props["schema"])
  df = df.withColumn("ingestion_time", lit(run_timestamp))
  df.write\
    .format("delta")\
    .mode("append")\
    .option("overwriteSchema", "true")\
    .save(f"{config["output"]["bronze_path"]}/{source}")


#Customers load

In [0]:


trasactions_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("transaction_date", StringType(), False),
    StructField("transaction_time", StringType(), False),
    StructField("transaction_type", StringType(), False),
    StructField("merchant_category", StringType(), True),
    StructField("merchant_subcategory", StringType(), True),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("location", StructType([
        StructField("province", StringType(), True),
        StructField("city", StringType(), True),
        StructField("coordinates", StringType(), True)
    ]), True),
    StructField("metadata", StructType([
        StructField("device_id", StringType(), True),
        StructField("session_id", StringType(), True),
        StructField("retry_flag", BooleanType(), True),
    ]), True)
  ])

run_timestamp = datetime.now()

df = spark.read.json("/Volumes/nedbank_ovation/bronze/raw_data/transactions.jsonl", schema=trasactions_schema)

df = df.withColumn("ingestion_time", lit(run_timestamp))

(
  df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nedbank_ovation.bronze.transactions")
)


In [0]:
%sql
SELECT * FROM nedbank_ovation.bronze.transactions